# 🎬 Promo Extractor (Kuku · Problem 1)
Finds 20–60s promo clips in a microdrama screenplay: a hook in the first 3s, a cliffhanger at the cut, and it makes sense to a stranger.

**Pipeline:** parse (Python) → Hook Hunter (LLM, 1 call per episode) → 60s window (Python) → Judge & Trimmer (LLM, 1 call per clip) + timestamp checks (Python) → tiers + Ranker (LLM, 1 call) → hand-check sheet.

**Before you start:** add your OpenAI API key in the 🔑 *Secrets* panel (left sidebar) as `OPENAI_API_KEY` and turn on *Notebook access*. Then run the cells in order.

## 1. Get the code

In [ ]:
REPO = "https://github.com/ShubNoob/KUKU_PROM"   # <- your repo

import glob, os, shutil
%cd /content
# keep any scripts zip uploaded earlier (re-downloading the repo would delete it)
for z in glob.glob("/content/repo/**/*.zip", recursive=True):
    shutil.copy(z, "/content/")
!rm -rf repo && git clone -q $REPO repo
hits = glob.glob("/content/repo/**/run.py", recursive=True)
assert hits, "run.py not found - check the files are on GitHub"
%cd {os.path.dirname(hits[0])}
!grep -q '_strict' llm_nodes.py && echo "ChatGPT version found" || echo "OLD CODE - upload the new files to GitHub first"
!pip -q install openai pydantic pyyaml

## 2. API key + check it works

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from openai import OpenAI
ids = sorted(m.id for m in OpenAI().models.list())
print("Key works. GPT models:", [m for m in ids if m.startswith("gpt")])

**Choose the model** using names from the list above. `FALLBACK` is used automatically if the main model is overloaded or rate-limited (`""` for none).
To switch provider (Claude, Gemini, ...), change `--provider` below and `provider` in `config.yaml`.

In [ ]:
MODEL = "gpt-5-mini"
FALLBACK = "gpt-4o-mini"
print("Using", MODEL, "then", FALLBACK or "no backup")

## 3. Episode scripts (.zip of .txt files)
Uses a zip already in `/content` if there is one, otherwise asks you to upload it.

In [ ]:
import glob, os, shutil, zipfile
from google.colab import files

def episode_count(path):
    """How many episode .txt scripts a zip holds (0 if it is not a scripts zip)."""
    try:
        names = zipfile.ZipFile(path).namelist()
    except (zipfile.BadZipFile, FileNotFoundError):
        return 0
    return sum(n.lower().endswith(".txt") and "episode" in os.path.basename(n).lower()
               and "__macosx" not in n.lower() for n in names)

zips = [z for z in glob.glob("/content/*.zip") if episode_count(z) >= 5]
if zips:
    ZIP = zips[0]
else:
    print("No scripts zip found. Upload the zip of episode .txt files (NOT the code zip):")
    name = next(iter(files.upload()))
    ZIP = shutil.copy(os.path.abspath(name), "/content/")   # keep it safe in /content
    assert episode_count(ZIP) >= 5, f"{name} has no episode scripts - wrong zip?"
print(f"Scripts: {ZIP} ({episode_count(ZIP)} episode files)")

## 4. Sanity checks (no API calls, free)
Unit tests for the timestamp checks and model handling, then an offline dry run with the keyword *mock* model.

In [ ]:
!python test_checks.py | tail -1
!python run.py "$ZIP" --provider mock --out outputs_mock | head -4

## 5. Quick real run on 3 episodes
Read a couple of promos against the script before running the full show.

In [ ]:
!python run.py "$ZIP" --provider openai --model $MODEL --fallback "$FALLBACK" --episodes 1-3 --out outputs_test
print(open("outputs_test/shortlist.md").read())

## 6. Full run on all episodes
If it stops partway, run this cell again: finished calls are saved in `.cache/` and not repeated.

In [ ]:
!python run.py "$ZIP" --provider openai --model $MODEL --fallback "$FALLBACK" --out outputs

## 7. Results

In [ ]:
from IPython.display import Markdown, display
import json
log = json.load(open("outputs/run_log.json"))
print("Tiers:", log["tier_counts"], "| tokens in/out:", log["llm_stats"]["input_tokens"], "/", log["llm_stats"]["output_tokens"])
display(Markdown(open("outputs/shortlist.md").read()))

In [ ]:
import pandas as pd
pd.read_csv("outputs/promos.csv").head(20)

## 8. Download the raw outputs and the hand-check sheet

In [ ]:
!zip -qr raw_outputs.zip outputs
files.download("raw_outputs.zip")
files.download("outputs/hand_check.csv")

## 9. Hand check (human in the loop)
Open `hand_check.csv`, read each clip in the script, fill `human_good` (y/n) and `human_reason`, save as CSV, then upload it here.
The sheet hides the model's scores so you judge first.

In [ ]:
labelled = next(iter(files.upload()))
!python run.py --score "$labelled"